# Tutorial 09 — Train, evaluate, and track a fraud model

**Goal.** Turn a 10,000-payment simulation into a leakage-safe model comparison with visible metrics, a saved artifact, and a reproducibility manifest.

**Prerequisites.** `poetry install -E ml` and a local checkout. This tutorial runs offline and writes only to `runs/tutorial-09`.

**Produces.** PIT rows, schema/missingness tables, logistic and deterministic-heuristic scores, PR-AUC/ROC-AUC/recall-at-FPR/calibration/monetary metrics, a model artifact, and JSON manifests.


In [ ]:
from pathlib import Path

import polars as pl

from fraudtwin import generate
from fraudtwin.config import load_config
from fraudtwin.ml import (
    PointInTimeDatasetBuilder,
    load_baseline_config,
    train_baselines,
    write_evaluation,
)

root = Path.cwd()
out = root / "runs" / "tutorial-09"
config = load_config(root / "configs" / "benchmarks" / "m13-camouflage-v1.yaml")
values = config.model_dump(mode="python")
values["simulation"]["duration_days"] = 7
values["payments"]["daily_target"] = 1400
config = type(config).model_validate(values)
run = generate(config, write=True, output_dir=out)
print("run:", run.manifest.run_id, "payments:", len(run.behavior.payments))

## Inspect the data before training

Always inspect grain, nulls, class balance, and temporal boundaries before fitting. A row is usable only when its features were observable at `prediction_time`; labels may mature later.


In [ ]:
dataset = PointInTimeDatasetBuilder(config, run.entities, run.behavior, run.manifest).build()
rows = dataset.rows
frame = pl.DataFrame(rows)
print(frame.shape)
display(frame.head(8))
display(frame.null_count())
display(
    frame.group_by("split").agg(pl.len().alias("rows"), pl.col("label").mean().alias("fraud_rate"))
)
assert frame["prediction_time"].min() < frame["prediction_time"].max()

## Train on time, not random rows

The configured `train`, `validation`, and `test` periods are chronological. Compare models on the held-out test window, then inspect segment metrics to find performance that averages hide.


In [ ]:
policy = load_baseline_config(root / "configs" / "ml-baselines.yaml")
result = train_baselines(rows, policy, source_run_dir=out)
evaluation_dir = out / "evaluation"
predictions_path, metrics_path, manifest_path = write_evaluation(result, evaluation_dir)
metrics = pl.read_ndjson(metrics_path)
display(metrics)
print("artifact models:", list(result.model_artifacts or {}))
print("manifest:", manifest_path)

## Track the decision

If `MLFLOW_TRACKING_URI` is configured, log the same parameters, metrics, and artifact to MLflow. Without it, the local manifest remains complete and portable. Record the selected model, threshold policy, feature allowlist, and source fingerprint in your model registry change.


In [ ]:
import json

manifest = json.loads(manifest_path.read_text())
assert manifest["lineage"]["simulation_manifest_hash"]
assert manifest["output_fingerprint"]
assert any("test" in str(key).lower() for key in manifest["metrics"])
print(
    json.dumps(
        {
            "rows": len(rows),
            "models": manifest["models"],
            "fingerprint": manifest["output_fingerprint"],
        },
        indent=2,
    )
)

**Expected outcome.** A metrics table and artifact under `runs/tutorial-09/evaluation`, with a manifest that can be diffed in code review.

**Cleanup.** Remove `runs/tutorial-09` when finished. Next: [Tutorial 10 — promote and serve the model](10-stress-drift-and-camouflage.ipynb) and the [model lifecycle guide](../model-lifecycle.md).
